In [ ]:
import os 
import polars as pl
import simple_icd_10_cm as cm
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from tqdm import tqdm

from note_workers import note_pool_context, read_json_text_file

DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/')
OncDRS_PATH = Path('/data/gusev/PROFILE/CLINICAL/OncDRS/')
PROFILE_DATA_PATH = DATA_PATH / 'PROFILE_DATA/'
PROFILE_NOTES_PATH = PROFILE_DATA_PATH / 'CLINICAL_NOTES/'
os.makedirs(PROFILE_NOTES_PATH, exist_ok=True)

# Each worker reads and parses one JSON file at a time -- small, independent
# units of work -- so parallelism belongs at the process level. Workers run a
# 1-thread polars so that N workers x a full-width thread pool each does not
# oversubscribe the node; see note_workers.note_pool_context for why that cap
# has to be set before the pool is built rather than in an initializer.
MAX_WORKERS = min(16, os.cpu_count() or 4)
WORKER_THREADS = 1
MP_CONTEXT = note_pool_context(WORKER_THREADS)

def note_pool():
    return ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        mp_context=MP_CONTEXT,
    )

def human_size(path: str | Path) -> str:
    size = Path(path).stat().st_size

    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024

text_pulls = {
    'CLINICAL_TEXTS_2024_03' : {'sub_dirs' : None}, 
    'CLINICAL_TEXTS_2025_03' : {'sub_dirs' : None},
    'CLINICAL_TEXTS_2025_11' : {'sub_dirs' : None}, 
    'CLINICAL_TEXTS_2026_03' : {
        'sub_dirs' : ['Discharge Summary Notes', 'Pathology_Cytology Notes', 
                      'Progress Notes', 'Imaging Notes']}
}

image_path_schema = {'RPT_ID' : pl.Int64,
                     'DFCI_MRN' : pl.Int64, 
                     'EVENT_DATE' : pl.String, 
                     'PROC_DESC' : pl.String, 
                     'RPT_TYPE' : pl.String,}

prog_disc_schema = {'RPT_ID' : pl.Int64,
                    'DFCI_MRN' : pl.Int64, 
                    'EVENT_DATE' : pl.String,
                    'INP_RPT_TYPE' : pl.String,
                    'PROVIDER_TYPE' : pl.String, 
                    'ENCOUNTER_TYPE_DESC' : pl.String,}

files_to_search = []

for pull, pull_map in text_pulls.items():
    PULL_PATH = OncDRS_PATH / pull

    if pull_map["sub_dirs"]:
        for sub_dir in pull_map["sub_dirs"]:
            files_to_search += sorted(
                PULL_PATH / sub_dir / f
                for f in os.listdir(PULL_PATH / sub_dir)
                if f.endswith(".json")
            )
    else:
        files_to_search += sorted(
            PULL_PATH / f
            for f in os.listdir(PULL_PATH)
            if f.endswith(".json")
        )
        
progress_files = [f for f in files_to_search if ('Prog' in str(f))]
discharge_files = [f for f in files_to_search if ('Discharge' in str(f))]
path_files = [f for f in files_to_search if ('Path' in str(f))]
image_files = [f for f in files_to_search if ('Imaging' in str(f))]

assert(len(progress_files + discharge_files + path_files + image_files) == len(files_to_search))

def compile_metadata(files, schema):
    """Read every file in `files` and concatenate them in list order.

    executor.map yields results in submission order, so the concatenated frame
    matches the serial ordering. That ordering is load-bearing: `files` runs
    oldest pull -> newest, and the caller's unique(keep='last') relies on it to
    let the newest pull win.
    """
    reader = partial(
        read_json_text_file,
        schema=schema,
        oncdrs_path=OncDRS_PATH,
    )

    with note_pool() as executor:
        frames = list(tqdm(executor.map(reader, files, chunksize=1), total=len(files)))

    return (pl.concat(frames, how='vertical')
            .with_columns(pl.col('EVENT_DATE').str.to_datetime(format="%Y-%m-%dT%H:%M:%SZ", time_zone='UTC')))

# get path_dfs
from time import time
start = time()
print(f'starting path files')

path_df = compile_metadata(path_files, image_path_schema)

deduped_path_df = path_df.unique(subset=['RPT_ID'], keep='last')
deduped_path_df.write_parquet(PROFILE_NOTES_PATH / 'PATHOLOGY_NOTES_METADATA.parquet', compression='zstd', compression_level=10)
end = time()
print(f'path meta compilation complete in {(end - start) / 60 : 0.2f} minutes')

start = time()
print(f'starting image files')

image_df = compile_metadata(image_files, image_path_schema)

deduped_image_df = image_df.unique(subset=['RPT_ID'], keep='last')
deduped_image_df.write_parquet(PROFILE_NOTES_PATH / 'IMAGING_NOTES_METADATA.parquet', compression='zstd', compression_level=10)
end = time()
print(f'image meta compilation complete in {(end - start) / 60 : 0.2f} minutes')

start = time()
print(f'starting progress/discharge files')

prog_df = compile_metadata(progress_files + discharge_files, prog_disc_schema)

deduped_prog_df = prog_df.unique(subset=['RPT_ID'], keep='last')
deduped_prog_df.write_parquet(PROFILE_NOTES_PATH / 'PROGRESS_NOTES_METADATA.parquet', compression='zstd', compression_level=10)
end = time()
print(f'progress/discharge meta compilation complete in {(end - start) / 60 : 0.2f} minutes')